# Comparison of LLM Judges Performance

## 1. Setup

In [1]:
import os, json, sys
import pandas as pd
from typing import Dict, List, Tuple, Optional
import argparse
from datetime import datetime

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sycophancy_analysis.api import SCORING_CONFIG
from sycophancy_analysis.scoring import (
    PromptMeta,
    PromptScores,
    score_response_llm,
    score_response,
)

# Model configurations with optimized settings
MODEL_CONFIGS = {
    "hermes-405b": {
        "model": "nousresearch/hermes-4-405b",
        "temperature": 0.0,
        "max_tokens": 1200,
        "provider_prefs": {"openrouter": 1.0}
    },
    "claude-sonnet": {
        "model": "anthropic/claude-3.5-sonnet",
        "temperature": 0.1,
        "max_tokens": 1000,
        "provider_prefs": {"anthropic": 1.0}
    },
    "gpt4-turbo": {
        "model": "openai/gpt-4-turbo-preview",
        "temperature": 0.0,
        "max_tokens": 1200,
        "provider_prefs": {"openai": 1.0}
    },
    "llama-405b": {
        "model": "meta-llama/llama-3.1-405b-instruct",
        "temperature": 0.2,
        "max_tokens": 1500,
        "provider_prefs": {"meta": 0.8, "together": 0.2}
    },
    "gemini-pro": {
        "model": "google/gemini-pro-1.5",
        "temperature": 0.1,
        "max_tokens": 1000,
        "provider_prefs": {"google": 1.0}
    },
    # Add more model configurations as needed
}

def setup_argparse():
    """Setup command line argument parsing"""
    parser = argparse.ArgumentParser(description="Evaluate sycophancy detection with specific model configurations")
    parser.add_argument(
        "--model", 
        type=str, 
        required=True,
        choices=list(MODEL_CONFIGS.keys()),
        help="Model configuration to use"
    )
    parser.add_argument(
        "--sample-n", 
        type=int, 
        default=140,
        help="Number of samples to evaluate (default: 140)"
    )
    parser.add_argument(
        "--random-seed", 
        type=int, 
        default=42,
        help="Random seed for sampling (default: 42)"
    )
    parser.add_argument(
        "--output-dir", 
        type=str, 
        default=None,
        help="Output directory for results (default: project_root/evaluation_results)"
    )
    parser.add_argument(
        "--verbose", 
        action="store_true",
        help="Enable verbose output"
    )
    return parser

def load_and_prepare_data(sample_n: int, random_seed: int, verbose: bool = False):
    """Load and prepare the evaluation dataset"""
    DATASET_DIR = os.path.join(project_root, "dataset")
    PROMPTS_JSON = os.path.join(DATASET_DIR, "prompt_battery.json")
    GROUND_TRUTH_CSV = os.path.join(DATASET_DIR, "ground_truth.csv")

    # Load prompts
    with open(PROMPTS_JSON, "r", encoding="utf-8") as f:
        prompts = json.load(f)

    prompts_df = pd.DataFrame(prompts)
    prompts_df["prompt_id"] = prompts_df["prompt_id"].astype(str)

    # Load ground truth
    all_gt_df = pd.read_csv(GROUND_TRUTH_CSV)
    all_gt_df["prompt_id"] = all_gt_df["prompt_id"].astype(str)

    # Sample data if needed
    if sample_n and sample_n > 0 and sample_n < len(all_gt_df):
        gt_df = all_gt_df.sample(n=sample_n, random_state=random_seed).reset_index(drop=True)
    else:
        gt_df = all_gt_df.copy().reset_index(drop=True)

    if verbose:
        print(f"Loaded prompts: {len(prompts_df)} | GT rows (sampled): {len(gt_df)}")

    # Build prompt metadata
    pmeta: Dict[str, PromptMeta] = {}
    for r in prompts_df.itertuples(index=False):
        s_val = int(getattr(r, "stance", 0))
        topic_harm_flag = bool(getattr(r, "is_harmful", False))
        pmeta[r.prompt_id] = PromptMeta(
            stance=s_val,
            strength=float(getattr(r, "strength", 0.0) or 0.0),
            is_harmful=(topic_harm_flag and (s_val == +1)),
            ask_devil=bool(getattr(r, "ask_devil", False)),
            topic=str(getattr(r, "topic", "misc")),
            persona=str(getattr(r, "persona", "neutral")),
            prompt_text=str(getattr(r, "text", "")),
        )

    return gt_df, pmeta

def configure_model(model_key: str, verbose: bool = False):
    """Configure the scoring system for a specific model"""
    config = MODEL_CONFIGS[model_key]
    
    # Update scoring configuration
    SCORING_CONFIG["USE_LLM_JUDGE"] = True
    SCORING_CONFIG["LLM_JUDGE_MODEL"] = config["model"]
    SCORING_CONFIG["LLM_JUDGE_TEMPERATURE"] = config["temperature"]
    SCORING_CONFIG["LLM_JUDGE_MAX_TOKENS"] = config["max_tokens"]
    SCORING_CONFIG["LLM_JUDGE_PROVIDER_PREFS"] = config["provider_prefs"]
    
    if verbose:
        print(f"Configured model: {config['model']}")
        print(f"  Temperature: {config['temperature']}")
        print(f"  Max tokens: {config['max_tokens']}")
        print(f"  Provider preferences: {config['provider_prefs']}")
    
    return config

def evaluate_responses(gt_df: pd.DataFrame, pmeta: Dict[str, PromptMeta], 
                      model_config: Dict, api_key: str, verbose: bool = False):
    """Evaluate responses using the configured model"""
    rows: List[Dict] = []
    total_rows = len(gt_df)
    
    for i, r in enumerate(gt_df.itertuples(index=False)):
        if verbose and i % 20 == 0:
            print(f"Processing row {i+1}/{total_rows}")
            
        pid = getattr(r, "prompt_id")
        meta = pmeta.get(pid)
        if meta is None:
            continue
            
        resp = str(getattr(r, "response", ""))

        try:
            s: PromptScores = score_response_llm(resp=resp, meta=meta, api_key=api_key)
        except Exception as e:
            if verbose:
                print(f"Error scoring row {i+1}: {e}")
            # Use heuristic fallback
            s = score_response(resp, meta)

        rows.append({
            "model": getattr(r, "model"),
            "prompt_id": pid,
            "human_eval": getattr(r, "human_eval"),
            "judge_model": model_config["model"],
            "judge_temperature": model_config["temperature"],
            "judge_max_tokens": model_config["max_tokens"],
            **s.__dict__,
        })

    return pd.DataFrame(rows)

def calculate_metrics(scored_df: pd.DataFrame, verbose: bool = False):
    """Calculate evaluation metrics"""
    # Filter to evaluable rows
    eval_subset = scored_df[scored_df["human_eval"].isin(["EVASION", "CHALLENGE", "AGREEMENT"])].copy()
    
    if len(eval_subset) == 0:
        return {
            "overall_accuracy": 0.0,
            "model_accuracies": {},
            "confusion_matrix": pd.DataFrame(),
            "total_samples": len(scored_df),
            "evaluated_samples": 0
        }
    
    # Overall accuracy
    overall_acc = float((eval_subset["human_eval"] == eval_subset["pred_label"]).mean())
    
    # Per-model accuracy
    model_accuracies = {}
    for model_name in eval_subset["model"].unique():
        model_subset = eval_subset[eval_subset["model"] == model_name]
        if len(model_subset) > 0:
            model_acc = (model_subset["human_eval"] == model_subset["pred_label"]).mean()
            model_accuracies[model_name] = float(model_acc)
    
    # Confusion matrix
    confusion_matrix = eval_subset.groupby(["human_eval", "pred_label"]).size().unstack(fill_value=0)
    
    # Per-category metrics
    category_metrics = {}
    for category in ["EVASION", "CHALLENGE", "AGREEMENT"]:
        cat_subset = eval_subset[eval_subset["human_eval"] == category]
        if len(cat_subset) > 0:
            correct = (cat_subset["human_eval"] == cat_subset["pred_label"]).sum()
            total = len(cat_subset)
            category_metrics[category] = {
                "accuracy": float(correct / total),
                "count": total,
                "correct": int(correct)
            }
    
    if verbose:
        print(f"Overall accuracy: {overall_acc:.3f}")
        print(f"Evaluated samples: {len(eval_subset)}/{len(scored_df)}")
        print("\nPer-model accuracies:")
        for model, acc in model_accuracies.items():
            print(f"  {model}: {acc:.3f}")
    
    return {
        "overall_accuracy": overall_acc,
        "model_accuracies": model_accuracies,
        "confusion_matrix": confusion_matrix,
        "category_metrics": category_metrics,
        "total_samples": len(scored_df),
        "evaluated_samples": len(eval_subset)
    }

def save_results(scored_df: pd.DataFrame, metrics: Dict, model_key: str, 
                model_config: Dict, output_dir: str, timestamp: str):
    """Save evaluation results"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save detailed scores
    detailed_filename = f"{model_key}_detailed_scores_{timestamp}.csv"
    scored_df.to_csv(os.path.join(output_dir, detailed_filename), index=False)
    
    # Save summary metrics
    summary_filename = f"{model_key}_summary_{timestamp}.json"
    summary_data = {
        "model_config": model_config,
        "evaluation_timestamp": timestamp,
        "metrics": {
            "overall_accuracy": metrics["overall_accuracy"],
            "total_samples": metrics["total_samples"],
            "evaluated_samples": metrics["evaluated_samples"],
            "model_accuracies": metrics["model_accuracies"],
            "category_metrics": metrics["category_metrics"]
        },
        "confusion_matrix": metrics["confusion_matrix"].to_dict() if not metrics["confusion_matrix"].empty else {}
    }
    
    with open(os.path.join(output_dir, summary_filename), "w") as f:
        json.dump(summary_data, f, indent=2, default=str)
    
    return detailed_filename, summary_filename

# Evaluation of LLM Judges

In [ ]:
"""Main execution function"""
parser = setup_argparse()
args = parser.parse_args()

# Setup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = args.output_dir or os.path.join(project_root, "evaluation_results")
api_key = os.environ.get("OPENROUTER_API_KEY", "")

if not api_key:
    print("Warning: No OPENROUTER_API_KEY found. Will use heuristic scoring as fallback.")

print(f"=== Evaluating Model: {args.model} ===")
print(f"Timestamp: {timestamp}")
print(f"Sample size: {args.sample_n}")
print(f"Output directory: {output_dir}")
print(f"API key present: {bool(api_key)}")
print()

# Load data
print("Loading dataset...")
gt_df, pmeta = load_and_prepare_data(args.sample_n, args.random_seed, args.verbose)

# Configure model
print("Configuring model...")
model_config = configure_model(args.model, args.verbose)
print()

# Run evaluation
print("Running evaluation...")
scored_df = evaluate_responses(gt_df, pmeta, model_config, api_key, args.verbose)
print(f"Scored {len(scored_df)} responses")
print()

# Calculate metrics
print("Calculating metrics...")
metrics = calculate_metrics(scored_df, args.verbose)
print()

# Print summary
print("=== EVALUATION RESULTS ===")
print(f"Model: {model_config['model']}")
print(f"Overall Accuracy: {metrics['overall_accuracy']:.3f}")
print(f"Samples Evaluated: {metrics['evaluated_samples']}/{metrics['total_samples']}")

if metrics['model_accuracies']:
    print("\nPer-Target-Model Accuracy:")
    for model, acc in metrics['model_accuracies'].items():
        print(f"  {model}: {acc:.3f}")

if metrics['category_metrics']:
    print("\nPer-Category Performance:")
    for category, cat_metrics in metrics['category_metrics'].items():
        print(f"  {category}: {cat_metrics['accuracy']:.3f} ({cat_metrics['correct']}/{cat_metrics['count']})")

if not metrics['confusion_matrix'].empty:
    print("\nConfusion Matrix (rows=Ground Truth, cols=Predicted):")
    print(metrics['confusion_matrix'])

# Save results
print(f"\nSaving results to {output_dir}...")
detailed_file, summary_file = save_results(
    scored_df, metrics, args.model, model_config, output_dir, timestamp
)
print(f"  Detailed scores: {detailed_file}")
print(f"  Summary metrics: {summary_file}")

print("\n=== EVALUATION COMPLETE ===")
